# Assignment 2: Data Cleaning & Preprocessing
**Dataset:** Telco Customer Churn (Kaggle, BlastChar)  
**Goal:** Data ko saaf karna aur model ke liye tayyar karna (cleaning, split, encoding, scaling).

**Steps:**
1. Data load aur inspect karna
2. Missing values (`TotalCharges`)
3. Duplicates check
4. Categorical columns ki safai
5. Outliers check
6. Target encode + Train/Test split
7. Skewness (log transform)
8. Encoding (Ordinal + One-Hot)
9. Scaling
10. Final checks aur save

## 1. Data load karna

In [ ]:
from google.colab import files
uploaded = files.upload()   # WA_Fn-UseC_-Telco-Customer-Churn.csv select karein

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
print("Shape:", df.shape)
df.head()

In [ ]:
df.info()

## 2. Missing values: `TotalCharges`
`TotalCharges` ka type `object` hai, yani is mein kuch values numbers nahi hain (khali space `' '` hai).  
`pd.to_numeric(..., errors='coerce')` aisi values ko `NaN` bana deta hai.

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print("Missing TotalCharges:", df['TotalCharges'].isnull().sum())

# Check: kya yeh sab naye customers hain (tenure = 0)?
df.loc[df['TotalCharges'].isnull(), ['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']]

Yeh 11 customers naye hain (`tenure = 0`), unka abhi tak koi bill nahi bana. Is liye `TotalCharges` ko **0** se bharna sahi hai (median se bharna galat hota, kyunke unhon ne abhi kuch pay hi nahi kiya).

In [ ]:
# Safety check: sirf wahi rows missing honi chahiye jin ki tenure 0 hai
assert (df.loc[df['TotalCharges'].isnull(), 'tenure'] == 0).all()

df['TotalCharges'] = df['TotalCharges'].fillna(0)
print("Missing values ab:\n", df.isnull().sum().sum())

## 3. Duplicates check

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate customerID:", df['customerID'].duplicated().sum())

Koi duplicate nahi mila, is liye kuch hatane ki zaroorat nahi.

## 4. Categorical columns ki safai
Pehle har column ki unique values dekhte hain, taake galat ya dohre labels pakde jayein.

In [ ]:
# Extra spaces hata dein (agar hon)
cat_cols = df.select_dtypes(include='object').columns.drop('customerID')
for c in cat_cols:
    df[c] = df[c].str.strip()

for c in cat_cols:
    print(c, "->", df[c].unique())

Kuch columns mein `'No internet service'` / `'No phone service'` aata hai. Yeh asal mein `'No'` hi ka matlab dete hain (service hai hi nahi), is liye inhein `'No'` bana dete hain. Isse one-hot encoding mein faltu columns nahi banenge. (Internet hai ya nahi, yeh info `InternetService` column mein pehle se maujood hai.)

In [ ]:
internet_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                 'TechSupport', 'StreamingTV', 'StreamingMovies']

for c in internet_cols:
    df[c] = df[c].replace('No internet service', 'No')

df['MultipleLines'] = df['MultipleLines'].replace('No phone service', 'No')

# Check: ab in columns mein sirf Yes/No hona chahiye
for c in internet_cols + ['MultipleLines']:
    print(c, df[c].unique())

`SeniorCitizen` pehle se 0/1 mein hai (yani already numeric binary), is liye isay encode karne ki zaroorat nahi. Isay seedha final features mein shamil karenge.

## 5. Outliers check (IQR method)

In [ ]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

df[num_cols].boxplot(figsize=(8, 4))
plt.title("Numerical columns ka boxplot")
plt.show()

for c in num_cols:
    Q1, Q3 = df[c].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    n_out = ((df[c] < lower) | (df[c] > upper)).sum()
    print(f"{c}: {n_out} outliers (range {lower:.1f} se {upper:.1f})")

**Faisla:** Agar upar outliers ki tadaad 0 hai to kisi treatment ki zaroorat nahi. Agar kuch outliers aayein to unhein **rakhna** behtar hai, kyunke yeh real customers hain (bohot zyada bill wale), galti nahi.

## 6. Target encode aur Train/Test split
`Churn` ko 0/1 banate hain. **Split pehle** karte hain aur encoding/scaling baad mein, taake test data ki info train mein leak na ho (data leakage).

In [ ]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
print(df['Churn'].value_counts(normalize=True).round(3))   # class balance

In [ ]:
X = df.drop(['Churn', 'customerID'], axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
X_train = X_train.copy()
X_test = X_test.copy()

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Churn rate train:", y_train.mean().round(3), "| test:", y_test.mean().round(3))

## 7. Skewness aur log transform

In [ ]:
print("Skewness (train):")
print(X_train[['tenure', 'MonthlyCharges', 'TotalCharges']].skew().round(3))

In [ ]:
# TotalCharges right-skewed hai, log1p lagate hain (log1p isliye ke 0 par bhi chalta hai)
X_train['TotalCharges_log'] = np.log1p(X_train['TotalCharges'])
X_test['TotalCharges_log']  = np.log1p(X_test['TotalCharges'])

print("Skewness pehle :", X_train['TotalCharges'].skew().round(3))
print("Skewness baad  :", X_train['TotalCharges_log'].skew().round(3))

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
X_train['TotalCharges'].hist(ax=ax[0], bins=30);     ax[0].set_title("TotalCharges (pehle)")
X_train['TotalCharges_log'].hist(ax=ax[1], bins=30); ax[1].set_title("TotalCharges (log ke baad)")
plt.show()

## 8. Encoding
- **Contract** ek *ordered* column hai (Month-to-month < One year < Two year), is liye Ordinal Encoding.
- Baqi categorical columns *unordered* hain, is liye One-Hot Encoding.

Encoders sirf **train** par `fit` hote hain, test par sirf `transform`.

In [ ]:
# --- Ordinal encoding: Contract ---
ord_enc = OrdinalEncoder(categories=[['Month-to-month', 'One year', 'Two year']])
X_train['Contract_encoded'] = ord_enc.fit_transform(X_train[['Contract']]).ravel()
X_test['Contract_encoded']  = ord_enc.transform(X_test[['Contract']]).ravel()

In [ ]:
# --- One-Hot encoding ---
onehot_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
               'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
               'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling',
               'PaymentMethod']

ohe = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)

train_ohe = ohe.fit_transform(X_train[onehot_cols])
test_ohe  = ohe.transform(X_test[onehot_cols])

ohe_names = ohe.get_feature_names_out(onehot_cols)
train_ohe_df = pd.DataFrame(train_ohe, columns=ohe_names, index=X_train.index)
test_ohe_df  = pd.DataFrame(test_ohe,  columns=ohe_names, index=X_test.index)

In [ ]:
# --- Final features jorna ---
# SeniorCitizen already 0/1 hai, isay seedha shamil karte hain
base_cols = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges_log', 'Contract_encoded']

X_train_final = pd.concat([X_train[base_cols], train_ohe_df], axis=1)
X_test_final  = pd.concat([X_test[base_cols],  test_ohe_df],  axis=1)

print(X_train_final.shape, X_test_final.shape)
X_train_final.head()

## 9. Scaling (StandardScaler)
Sirf continuous/ordinal columns scale karte hain. One-Hot aur `SeniorCitizen` pehle se 0/1 hain, unhein scale karne ki zaroorat nahi. Scaler bhi sirf train par `fit` hota hai.

In [ ]:
scale_cols = ['tenure', 'MonthlyCharges', 'TotalCharges_log', 'Contract_encoded']

scaler = StandardScaler()
X_train_final[scale_cols] = scaler.fit_transform(X_train_final[scale_cols])
X_test_final[scale_cols]  = scaler.transform(X_test_final[scale_cols])

X_train_final[scale_cols].describe().loc[['mean', 'std']].round(3)

## 10. Final checks aur save

In [ ]:
assert X_train_final.isnull().sum().sum() == 0
assert X_test_final.isnull().sum().sum() == 0
assert list(X_train_final.columns) == list(X_test_final.columns)
assert len(X_train_final) == len(y_train) and len(X_test_final) == len(y_test)

print("Train:", X_train_final.shape, "| Test:", X_test_final.shape)
print("Saare checks pass.")
X_train_final.dtypes.value_counts()

In [ ]:
# Cleaned data save karein (EDA / modeling ke liye)
X_train_final.to_csv("X_train_final.csv", index=False)
X_test_final.to_csv("X_test_final.csv", index=False)
y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)
df.to_csv("telco_cleaned.csv", index=False)

# Colab se download karne ke liye (optional):
# from google.colab import files
# files.download("telco_cleaned.csv")

## Summary
| Masla | Hal |
|---|---|
| `TotalCharges` text type, 11 khali values | Numeric banaya, naye customers ki wajah se 0 se bhara |
| `No internet service` / `No phone service` | `No` mein badal diya |
| Outliers | IQR se check kiye, decision upar likha |
| Data leakage ka khatra | Split pehle, encoders/scaler sirf train par fit |
| Skewed `TotalCharges` | `log1p` transform |
| Categorical columns | Ordinal (Contract) + One-Hot (baqi) |
| Scaling | StandardScaler (sirf continuous columns) |